In [1]:
# from transformers import BertForSequenceClassification, BertTokenizer, pipeline

# # Load the fine-tuned BERT-Emotion model
# model_name = "boltuix/bert-emotion"
# tokenizer = BertTokenizer.from_pretrained(model_name)
# model = BertForSequenceClassification.from_pretrained(model_name, num_labels=13)

In [ ]:
# model = model.to("mps")

In [31]:
# logits = model(**tokenizer("I love you", return_tensors="pt", truncation=True).to("mps"))
# label = logits.logits.argmax().item()
# text_label = model.config.id2label[label]
# print(text_label)

love


In [2]:
import textwrap
def pprint(text):
    print(textwrap.fill(str(text), 100))
    
from openai import OpenAI
import os
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ["OPENROUTER_API_KEY"],
)

In [3]:
import json
with open("../data/all_chats.json", "r") as f:
    data = json.load(f)

In [5]:
# Analyze emotion
import tqdm
import time
import torch
embeddings = []
batch_size = 64
for i in tqdm.tqdm(range(0, len(data), batch_size)):
    batch = data[i:i+batch_size]
    batch = [f"title: none | text: {text}" for text in batch]
    # logits = model(**tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("mps"))
    # labels = logits.logits.argmax(dim=1).cpu().tolist()
    # for text, label in zip(batch, labels):
    #     if model.config.id2label[label] == "fear":
    #         fears.append(text)
    while 1:
        try:
            embedding = client.embeddings.create(
                model="google/gemini-embedding-2-preview",
                input=batch,
                encoding_format="float"
            )
            embeddings.extend(embedding.data)
            break
        except Exception as e:
            print(f"Error: {e}. Retrying...")
            time.sleep(1)

100%|██████████| 242/242 [10:12<00:00,  2.53s/it]


In [19]:
import pickle
with open("embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

In [20]:
len(embeddings)

15440

In [21]:
import pickle
with open("embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

In [23]:
# embeddings = [embeddings[i].embedding for i in range(len(embeddings))]

In [24]:
embedding = client.embeddings.create(
    model="google/gemini-embedding-2-preview",
    input="task: search result | query: a question about worries on health",
    encoding_format="float"
)

In [25]:
import numpy as np
dataset = np.array(embeddings)
query = np.array(embedding.data[0].embedding)

In [26]:
dataset = dataset / np.linalg.norm(dataset, axis=1, keepdims=True)
query = query / np.linalg.norm(query)

In [27]:
idx = (query @ dataset.T).argsort()[::-1]

In [231]:
pprint(data[idx[190]])  

what are some potential cases where a overly cautious patient could end up being harmful or lethal
for themselves


In [ ]:
# correct but still anxious: listing symptons, listing harms, etc